# Node Testing and Mocking — fasteval-langgraph

Test individual nodes in isolation and mock dependencies with the harness's built-in mocking API.

**What you'll learn:**

1. **Single node execution** — `.node("name").run(**state)`
2. **NodeResult** — updates, goto, response, timing
3. **Mocking nodes** — `mock()` fluent builder
4. **Three patterns** — in-place, immutable, scoped context manager
5. **Dynamic mocks** — `.fn()` handler for state-dependent mocks
6. **Mocks + metrics** — Combine mocks with fasteval evaluation

Every cell is **runnable**. LLM-as-judge metrics require an OpenAI API key.

---

## Setup

In [ ]:
!pip install -q fasteval-langgraph

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab Secrets.")
except (ImportError, Exception):
    if os.environ.get("OPENAI_API_KEY"):
        print("API key found in environment.")
    else:
        print(
            "WARNING: OPENAI_API_KEY not set.\n"
            "LLM-as-judge metrics will fail. Node execution and mocking still work."
        )

In [ ]:
import fasteval as fe
from fasteval_langgraph import harness, mock
from fasteval_langgraph.models import NodeResult

print("Ready.")

---

## Build the Router Agent

We'll use a multi-node router agent with classifier, RAG, planner, and responder nodes.

> **Collab Note:** This is the same agent from the overview notebook, reused here for mocking demonstrations.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from langchain_core.messages import AIMessage, HumanMessage


class RouterState(MessagesState):
    intent: str
    docs: list
    plan: str


def classifier_node(state):
    last_msg = state["messages"][-1].content
    intent = "TROUBLESHOOTING" if "troubleshoot" in last_msg.lower() else "FAQ"
    return Command(update={"intent": intent}, goto="rag" if intent == "FAQ" else "planner")


def rag_node(state):
    return Command(update={"docs": [f"doc for: {state['intent']}"]}, goto="responder")


def planner_node(state):
    return Command(update={"plan": f"plan for: {state['intent']}"}, goto="responder")


def responder_node(state):
    context = state.get("docs") or [state.get("plan", "")]
    return {"messages": [AIMessage(content=f"Answer: {context[0]}")]}


rb = StateGraph(RouterState)
rb.add_node("classifier", classifier_node)
rb.add_node("rag", rag_node)
rb.add_node("planner", planner_node)
rb.add_node("responder", responder_node)
rb.add_edge(START, "classifier")
rb.add_edge("responder", END)
router_compiled = rb.compile(checkpointer=MemorySaver())

graph = harness(router_compiled)
print(f"Nodes: {graph.nodes}")

---

## 1. Single Node Execution

Use `.node("name").run(**state)` to execute one node without running the full graph.

> **Collab Note:** This is the fastest way to test routing logic, classification, or any individual node function.

In [ ]:
# Run the classifier in isolation
result = await graph.node("classifier").run(
    messages=[HumanMessage(content="What is OAuth?")]
)

print(f"Node:     {result.node_name}")
print(f"Updates:  {result.updates}")
print(f"Goto:     {result.goto}")
print(f"Time:     {result.execution_time_ms:.2f}ms")

In [ ]:
# Test routing decisions
faq_result = await graph.node("classifier").run(
    messages=[HumanMessage(content="What is OAuth?")]
)
assert faq_result.updates["intent"] == "FAQ"
assert faq_result.goto == "rag"
print(f"FAQ:            intent={faq_result.updates['intent']}, goto={faq_result.goto}")

ts_result = await graph.node("classifier").run(
    messages=[HumanMessage(content="I need to troubleshoot this")]
)
assert ts_result.updates["intent"] == "TROUBLESHOOTING"
assert ts_result.goto == "planner"
print(f"Troubleshoot:   intent={ts_result.updates['intent']}, goto={ts_result.goto}")

In [ ]:
# Node that produces an AI message
result = await graph.node("responder").run(
    messages=[HumanMessage(content="Test")],
    docs=["some retrieved document"],
)

print(f"Response: {result.response}")
assert result.response is not None
assert "some retrieved document" in result.response

In [ ]:
# Non-existent node raises ValueError
try:
    await graph.node("nonexistent").run(x=1)
except ValueError as e:
    print(f"Error: {e}")

---

## 2. Inspecting NodeResult

`NodeResult` is a Pydantic model with five fields:

| Field | Type | Description |
|-------|------|-------------|
| `node_name` | `str` | Name of the executed node |
| `updates` | `dict` | State updates returned by the node |
| `goto` | `Optional[str]` | Routing target (from `Command`) |
| `response` | `Optional[str]` | AI message content, if any |
| `execution_time_ms` | `float` | Execution time |

In [ ]:
result = await graph.node("classifier").run(
    messages=[HumanMessage(content="Hello")]
)

print(f"isinstance(result, NodeResult): {isinstance(result, NodeResult)}")
print(f"node_name:        {result.node_name}")
print(f"updates:          {result.updates}")
print(f"goto:             {result.goto}")
print(f"response:         {result.response}")
print(f"execution_time_ms: {result.execution_time_ms:.2f}")

---

## 3. Mocking Nodes

The harness provides a fluent `mock()` builder to replace nodes with fakes.

> **Collab Note:** Mocking is the key to isolated testing. Replace expensive LLM calls, flaky APIs, or slow databases with deterministic fakes while testing the surrounding logic.

In [ ]:
# Pattern 1: In-place mock (mutates the harness)
graph_copy = harness(router_compiled)
graph_copy.mock("rag").updates({"docs": ["in-place mock"]}).goto("responder")

result = await graph_copy.chat("What is OAuth?")
print(f"Docs: {result.state.get('docs')}")
assert "in-place mock" in str(result.state.get("docs", []))

# Clean up
graph_copy.reset_mocks()
print("Mocks reset.")

In [ ]:
# Pattern 2: Immutable copy (original unchanged)
original = harness(router_compiled)
test_graph = original.with_mocks(
    mock("rag").updates({"docs": ["mocked doc"]}).goto("responder"),
)

result = await test_graph.chat("What is OAuth?")
print(f"Test graph docs: {result.state.get('docs')}")
assert "mocked doc" in str(result.state.get("docs", []))

# Original is unaffected
test_graph.reset_mocks()
orig_result = await original.chat("What is OAuth?")
print(f"Original docs:   {orig_result.state.get('docs')}")
assert "mocked doc" not in str(orig_result.state.get("docs", []))

In [ ]:
# Pattern 3: Scoped context manager (auto-restores)
g = harness(router_compiled)

with g.mocked(
    mock("rag").updates({"docs": ["scoped mock"]}).goto("responder"),
):
    result = await g.chat("What is OAuth?")
    print(f"Inside scope: {result.state.get('docs')}")
    assert "scoped mock" in str(result.state.get("docs", []))

# After scope: automatically restored
result2 = await g.chat("What is OAuth?")
print(f"After scope:  {result2.state.get('docs')}")
assert "scoped mock" not in str(result2.state.get("docs", []))
print("\nContext manager auto-restored mocks.")

---

## 4. Dynamic Mocks

Use `.fn()` to compute mock results based on the input state.

> **Collab Note:** Dynamic mocks let you simulate realistic behavior (e.g., different results for different intents) while staying deterministic.

In [ ]:
g = harness(router_compiled)

with g.mocked(
    mock("rag")
    .fn(lambda state: {"docs": [f"dynamic doc for {state.get('intent', '?')}"]})
    .goto("responder"),
):
    result = await g.chat("What is OAuth?")
    docs = result.state.get("docs", [])
    print(f"Docs: {docs}")
    assert "dynamic doc for FAQ" in docs[0]

In [ ]:
# Dynamic mock with custom routing logic
def smart_rag_mock(state):
    intent = state.get("intent", "")
    if intent == "FAQ":
        return {"updates": {"docs": ["FAQ answer from mock"]}, "goto": "responder"}
    else:
        return {"updates": {"docs": ["Troubleshoot guide from mock"]}, "goto": "responder"}


with g.mocked(mock("rag").fn(smart_rag_mock)):
    faq_result = await g.chat("What is OAuth?")
    print(f"FAQ docs: {faq_result.state.get('docs')}")

In [ ]:
# Mock validation: non-existent node raises ValueError
try:
    g.with_mocks(mock("nonexistent").updates({"x": 1}))
except ValueError as e:
    print(f"Error: {e}")

---

## 5. Combining Mocks with fasteval Metrics

Mock upstream dependencies, then evaluate the node under test with fasteval.

> **Collab Note:** This is the most important pattern: isolate what you're testing, control the inputs, and use fasteval to judge the output quality.

In [ ]:
g = harness(router_compiled)


@fe.correctness(threshold=0.5)
async def test_responder_with_mocked_rag():
    """Test the responder with controlled RAG output."""
    with g.mocked(
        mock("rag")
        .updates({"docs": ["OAuth is an open standard for access delegation"]})
        .goto("responder"),
    ):
        result = await g.chat("What is OAuth?")
        return fe.score(
            actual_output=result.response,
            expected_output="OAuth is an authorization standard for access delegation",
            input="What is OAuth?",
        )


eval_result = await test_responder_with_mocked_rag()
print(f"Passed: {eval_result.passed}  |  Score: {eval_result.aggregate_score:.2f}")
for m in eval_result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

In [ ]:
# Test a single node with fasteval metrics

@fe.exact_match(threshold=1.0)
async def test_classifier_faq():
    """Verify classifier identifies FAQ correctly."""
    result = await graph.node("classifier").run(
        messages=[HumanMessage(content="What is OAuth?")]
    )
    return fe.score(
        actual_output=result.updates.get("intent", ""),
        expected_output="FAQ",
    )


eval_result = await test_classifier_faq()
print(f"Passed: {eval_result.passed}  |  Score: {eval_result.aggregate_score:.2f}")
for m in eval_result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

---

## When to Mock vs Test End-to-End

| Situation | Approach |
|-----------|----------|
| Testing routing logic | `.node().run()` — no mocks needed |
| Testing one node with controlled inputs | `.mocked()` upstream nodes |
| Testing the full agent flow | No mocks — `.chat()` or `.session()` |
| Flaky external dependencies | Mock the node that calls them |
| Expensive LLM calls in CI | Mock the LLM node, test the rest |

### Testing Pyramid

```
         /\
        /  \
       / E2E \         ← graph.chat() / graph.session()
      /--------\
     / Integration \    ← graph.mocked() / graph.with_mocks()
    /--------------\
   /   Unit Tests   \   ← graph.node().run()
  /------------------\
```

Start with node-level unit tests (fast, deterministic), add integration tests with mocks, and top off with E2E tests.

---

## Next Steps

| Notebook | What you'll learn |
|----------|------------------|
| [Harness Basics](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langgraph-testing/harness-basics.ipynb) | `.chat()`, `ChatResult`, hooks, auto-detection |
| [Multi-turn Sessions](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langgraph-testing/sessions.ipynb) | `.session()`, conversation metrics, interrupt/resume |